In [192]:
import sys
sys.path.append('..')
from osp import *

In [193]:
nlp_keys = set(STASH_SLICES_NLP.keys_l())

In [203]:
df_probs = pd.read_pickle('../data/raw/df_probs2.pkl.gz')#.drop_duplicates(subset=['id'])
df_probs = df_probs[df_probs.id.isin(nlp_keys)]

In [204]:
ids = df_probs.sample(1000).id.tolist()
# ids

def get_pos2words(ids):
    pos2words = defaultdict(list)
    ok_words = get_ok_words()

    for id in tqdm(ids):
        if id not in STASH_SLICES_NLP:
            continue
        doc = stanza.Document.from_serialized(STASH_SLICES_NLP[id])
        for sent in doc.sentences:
            for word in sent.words:
                pos = word.xpos
                text = word.text
                if (len(text)<2 and text not in {'a','I'}) or text.lower() not in ok_words:
                    continue
                feats = word.feats
                deprel = word.deprel
                key = (pos,feats,deprel)
                pos2words[key].append(text)
    return pos2words

In [213]:
pos2words = get_pos2words(ids[:1000])

100%|██████████| 1000/1000 [00:24<00:00, 41.47it/s]


In [214]:
# pos2words

In [215]:
def scramble_passage(doc, maxlen=500):
    out = []
    for sent in doc.sentences:
        for token in sent.words:
            tok = token.text
            upos = token.upos
            xpos = token.xpos
            feats = token.feats
            deprel = token.deprel
            key = (xpos,feats,deprel)
            newtok = random.choice(pos2words[key]) if key in pos2words and pos2words[key] else tok
            if tok[0]==tok[0].upper():
                newtok = newtok[0].upper() + newtok[1:]
            else:
                newtok = newtok[0].lower() + newtok[1:]
            space_after = token.to_dict().get('misc') != ""
            if upos == 'PUNCT':
                out.append(tok)
            else:
                out.append(newtok)
            if space_after:
                out.append(" ")
        if len("".join(out)) > maxlen:
            break
        out.append(" ")
    return "".join(out)
                

In [265]:
cmp='2000-2025 Philosophy vs 2000-2025 Other'
pred='2000-2025 Philosophy'
df = df_probs.query(f'comparison=="{cmp}" & pred=="{pred}"').sort_values('prob1',ascending=False)
df2 = df_probs.query(f'comparison=="{cmp}" & pred!="{pred}"').sort_values('prob2',ascending=False)
# df

In [268]:
def pprint2(s):
    import textwrap
    import textwrap
    for line in textwrap.wrap(s, break_long_words=True, replace_whitespace=False, break_on_hyphens=True):
        print(line)


In [324]:
def scramble_passage_i(df, i=0, feats=['deprel_mark', 'pos_MD', 'deprel_cop'], maxlen=1000):
    row = df.iloc[i]
    id = row.id
    featd = STASH_SLICE_FEATS[id]
    featd = {feat:featd.get(feat,0) for feat in feats}
    print(row)
    print(pd.Series(featd))
    doc = stanza.Document.from_serialized(STASH_SLICES_NLP[id])
    print()
    pprint2(doc.text[:maxlen])
    print()
    pprint2(scramble_passage(doc,maxlen=maxlen))
    # scramble_passage(doc)

In [325]:
scramble_passage_i(df, maxlen=500)

comparison    2000-2025 Philosophy vs 2000-2025 Other
id                            phil/10.1086/345626__02
prob1                                             1.0
prob2                                             0.0
pred                             2000-2025 Philosophy
Name: 2872722, dtype: object
deprel_mark    94.909405
pos_MD         25.884383
deprel_cop     75.927524
dtype: float64

When, therefore, I talk of anything I get as my own good, I must mean
either that the thing I get is good, or that my possessing it is good.
In both cases it is only the thing or the possession of it which is
mine, and not the goodness of that thing or that possession.
There is
no longer any meaning in attaching the my to our predicate and saying:
The possession of this by me is my good.
The good of it can in no
possible sense be private or belong to me; any more than a thing can
exist privately

When, thus, I conclude to anything I take in my undeveloped tripe, I
can make either for the narrative I des

In [334]:
scramble_passage_i(df2, 1,maxlen=500)

comparison    2000-2025 Philosophy vs 2000-2025 Other
id                          other/10.2307/3660878__06
prob1                                        0.000004
prob2                                        0.999996
pred                                  2000-2025 Other
Name: 2870474, dtype: object
deprel_mark    20.576132
pos_MD          3.292181
deprel_cop     10.699588
dtype: float64

The small herds did not compete with farms for the floodplain acreage,
and the livestock trade allowed the villagers to trade away extra
horses before the herds became ecologically unmanageable.
As a result,
their farming economy remained strong and crops voluminous.
Given the
northern climate, crop failures were strikingly rare, and their large
fields yielded enough corn for their own use, trade, and even for
their horses.
Essentially garrison communities, they were relatively
safe against fron

The conscious societies did not say for implications of the
normativity self, or the hearing presence discus

In [322]:
df_meta = get_corpus_metadata()

In [323]:
df_meta.loc['other/10.2307/3660878']

uuid                                              fe1583c5-6afb-3bed-8a39-f8530047510b
title                                The Rise and Fall of Plains Indian Horse Cultures
author                                                                Pekka Hämäläinen
year                                                                              2003
journal                                                The Journal of American History
volume                                                                              90
issue                                                                                3
url                                                   jstor.org/stable/10.2307/3660878
publisher                 Organization of American Historians; Oxford University Press
discipline                                                                       Other
discipline_names                                            History / American Studies
decade                                     

In [327]:
df_meta.loc['phil/10.1086/345626']

uuid                        da113c26-5124-380b-bca6-4aeba54fd3f1
title                     Neutral and Relative Value after Moore
author                                             Michael Smith
year                                                        2003
journal                                                   Ethics
volume                                                       113
issue                                                          3
url                              jstor.org/stable/10.1086/345626
publisher                        The University of Chicago Press
discipline                                            Philosophy
discipline_names                                             NaN
decade                                                      2000
period                                                 2000-2025
century                                                      C21
halfcentury                                                 eC21
century_discipline       